# День 5 — Fine-tuning DistilBERT

Цель: выполнить fine-tuning `distilbert-base-uncased`
для бинарной классификации тональности SST-2.

Используем фиксированные train и validation выборки,
сохранённые в День 4.

In [1]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer

C:\ProgramData\anaconda3\envs\transformers_overall_clean\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Device:", device)

PyTorch: 2.13.0+cpu
Device: cpu


In [3]:
data_dir = Path("../data")
splits_dir = data_dir / "splits"

train_path = splits_dir / "train.csv"
validation_path = splits_dir / "validation.csv"

train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)

display(train_df.head())

Train: (1600, 2)
Validation: (400, 2)


,text,label
0,better characters,1
1,a sturdiness and solidity that we 've long ass...,1
2,ill-wrought hypothesis,0
3,is a rambling examination of american gun cult...,0
4,do n't see often enough these days,1


In [4]:
print("Train classes:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation classes:")
print(validation_df["label"].value_counts().sort_index())

assert train_df.shape == (1600, 2)
assert validation_df.shape == (400, 2)

print("\nДанные загружены корректно")

Train classes:
label
0    800
1    800
Name: count, dtype: int64

Validation classes:
label
0    200
1    200
Name: count, dtype: int64

Данные загружены корректно


In [5]:
from torch.utils.data import Dataset

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer:", tokenizer.name_or_path)
print("Max length for our dataset:", MAX_LENGTH)

Tokenizer: distilbert-base-uncased
Max length for our dataset: 128


In [6]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }

In [7]:
train_dataset = SentimentDataset(
    texts=train_df["text"],
    labels=train_df["label"],
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

validation_dataset = SentimentDataset(
    texts=validation_df["text"],
    labels=validation_df["label"],
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(validation_dataset))

Train dataset: 1600
Validation dataset: 400


In [8]:
sample = train_dataset[0]

print("Keys:", sample.keys())
print("input_ids shape:", sample["input_ids"].shape)
print("attention_mask shape:", sample["attention_mask"].shape)
print("label:", sample["labels"])

print("Original text:")
print(train_df.iloc[0]["text"])

print("\nDecoded tokens:")
print(
    tokenizer.decode(
        sample["input_ids"],
        skip_special_tokens=True,
    )
)

Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids shape: torch.Size([128])
attention_mask shape: torch.Size([128])
label: tensor(1)
Original text:
better characters

Decoded tokens:
better characters


In [9]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

Train batches: 100
Validation batches: 25


In [10]:
batch = next(iter(train_loader))

print("Batch keys:", batch.keys())
print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("labels:", batch["labels"].shape)

print("\nLabels:")
print(batch["labels"])

Batch keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids: torch.Size([16, 128])
attention_mask: torch.Size([16, 128])
labels: torch.Size([16])

Labels:
tensor([1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1])


In [11]:
assert batch["input_ids"].shape == (BATCH_SIZE, MAX_LENGTH)
assert batch["attention_mask"].shape == (BATCH_SIZE, MAX_LENGTH)
assert batch["labels"].shape == (BATCH_SIZE,)

print("DataLoader работает корректно.")

DataLoader работает корректно.


In [12]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={
        0: "negative",
        1: "positive",
    },
    label2id={
        "negative": 0,
        "positive": 1,
    },
)

model = model.to(device)

print("Model:", MODEL_NAME)
print("Device:", next(model.parameters()).device)
print("Number of labels:", model.config.num_labels)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5912.22it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: distilbert-base-uncased
Device: cpu
Number of labels: 2


In [13]:
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

print("input_ids:", input_ids.shape)
print("attention_mask:", attention_mask.shape)
print("labels:", labels.shape)

input_ids: torch.Size([16, 128])
attention_mask: torch.Size([16, 128])
labels: torch.Size([16])


In [14]:
model.eval()

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )

print("Loss:", outputs.loss.item())
print("Logits shape:", outputs.logits.shape)
print("Logits:")
print(outputs.logits[:3])

Loss: 0.6946006417274475
Logits shape: torch.Size([16, 2])
Logits:
tensor([[-0.0023, -0.0229],
        [ 0.0010, -0.0261],
        [ 0.0307, -0.0066]])


In [15]:
predictions = torch.argmax(
    outputs.logits,
    dim=1,
)

print("Predictions:")
print(predictions)

print("\nTrue labels:")
print(labels)

Predictions:
tensor([0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1])

True labels:
tensor([1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1])


In [16]:
from torch.optim import AdamW

LEARNING_RATE = 2e-5

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
)

print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", LEARNING_RATE)

Optimizer: AdamW
Learning rate: 2e-05


In [17]:
model.train()

optimizer.zero_grad()

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels,
)

loss = outputs.loss

loss.backward()

torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0,
)

optimizer.step()

print("Training loss:", loss.item())

Training loss: 0.700333833694458


In [18]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={
        0: "negative",
        1: "positive",
    },
    label2id={
        "negative": 0,
        "positive": 1,
    },
)

model = model.to(device)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
)

print("Модель перезагружена")
print("Optimizer создан заново")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6154.25it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Модель перезагружена
Optimizer создан заново


In [19]:
def train_epoch(
    model,
    data_loader,
    optimizer,
    device,
):
    model.train()

    total_loss = 0.0

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0,
        )

        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(data_loader)

    return average_loss

In [20]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(
    model,
    data_loader,
    device,
):
    model.eval()

    total_loss = 0.0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )

            loss = outputs.loss
            logits = outputs.logits

            predictions = torch.argmax(
                logits,
                dim=1,
            )

            total_loss += loss.item()

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    average_loss = total_loss / len(data_loader)

    accuracy = accuracy_score(
        all_labels,
        all_predictions,
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
    )

    return average_loss, accuracy, macro_f1

In [21]:
fine_tuned_dir = Path("../models/fine_tuned_model")
output_dir = Path("../outputs/fine_tuned")

fine_tuned_dir.mkdir(
    parents=True,
    exist_ok=True,
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

print("Model directory:", fine_tuned_dir.resolve())
print("Output directory:", output_dir.resolve())

Model directory: C:\Users\User\Projects\transformers_overall_fix\models\fine_tuned_model
Output directory: C:\Users\User\Projects\transformers_overall_fix\outputs\fine_tuned


In [22]:
NUM_EPOCHS = 3

training_history = []

best_validation_f1 = -1.0
best_epoch = None

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"Epoch {epoch}/{NUM_EPOCHS}")

    train_loss = train_epoch(
        model=model,
        data_loader=train_loader,
        optimizer=optimizer,
        device=device,
    )

    (
        validation_loss,
        validation_accuracy,
        validation_f1,
    ) = evaluate(
        model=model,
        data_loader=validation_loader,
        device=device,
    )

    training_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": validation_loss,
        "validation_accuracy": validation_accuracy,
        "validation_macro_f1": validation_f1,
    })

    print(f"Train loss: {train_loss:.4f}")
    print(f"Validation loss: {validation_loss:.4f}")
    print(
        f"Validation accuracy: "
        f"{validation_accuracy:.4f}"
    )
    print(
        f"Validation macro F1: "
        f"{validation_f1:.4f}"
    )

    if validation_f1 > best_validation_f1:
        best_validation_f1 = validation_f1
        best_epoch = epoch

        model.save_pretrained(
            fine_tuned_dir,
            safe_serialization=True,
            max_shard_size="50MB",
        )

        tokenizer.save_pretrained(
            fine_tuned_dir
        )

        print(
            f"Новая лучшая модель сохранена "
            f"на эпохе {epoch}"
        )

    print()

Epoch 1/3
Train loss: 0.4477
Validation loss: 0.3258
Validation accuracy: 0.8875
Validation macro F1: 0.8875


Writing model shards: 100%|██████████| 5/5 [00:00<00:00, 20.48it/s]


Новая лучшая модель сохранена на эпохе 1

Epoch 2/3
Train loss: 0.2086
Validation loss: 0.3775
Validation accuracy: 0.8750
Validation macro F1: 0.8745

Epoch 3/3
Train loss: 0.1301
Validation loss: 0.4746
Validation accuracy: 0.8900
Validation macro F1: 0.8900


Writing model shards: 100%|██████████| 5/5 [00:00<00:00, 18.61it/s]

Новая лучшая модель сохранена на эпохе 3



In [23]:
history_df = pd.DataFrame(
    training_history
)

display(history_df)

history_path = (
    output_dir
    / "training_history.csv"
)

history_df.to_csv(
    history_path,
    index=False,
    encoding="utf-8",
)

print("Training history:", history_path.resolve())
print("Best epoch:", best_epoch)
print(
    "Best validation macro F1:",
    f"{best_validation_f1:.4f}",
)

,epoch,train_loss,validation_loss,validation_accuracy,validation_macro_f1
0,1,0.447659,0.325751,0.8875,0.887482
1,2,0.208629,0.377549,0.8750,0.874470
2,3,0.130092,0.474583,0.8900,0.889975


Training history: C:\Users\User\Projects\transformers_overall_fix\outputs\fine_tuned\training_history.csv
Best epoch: 3
Best validation macro F1: 0.8900


In [24]:
best_model = AutoModelForSequenceClassification.from_pretrained(
    fine_tuned_dir
)

best_model = best_model.to(device)

best_tokenizer = AutoTokenizer.from_pretrained(
    fine_tuned_dir
)

print("Лучшая модель загружена")
print("Device:", next(best_model.parameters()).device)
print("Number of labels:", best_model.config.num_labels)

(
    best_validation_loss,
    best_validation_accuracy_check,
    best_validation_f1_check,
) = evaluate(
    model=best_model,
    data_loader=validation_loader,
    device=device,
)

print(
    "Reloaded validation accuracy:",
    f"{best_validation_accuracy_check:.4f}",
)

print(
    "Reloaded validation macro F1:",
    f"{best_validation_f1_check:.4f}",
)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6246.62it/s]


Лучшая модель загружена
Device: cpu
Number of labels: 2
Reloaded validation accuracy: 0.8900
Reloaded validation macro F1: 0.8900


In [25]:
BASELINE_ACCURACY = 0.8525
BASELINE_MACRO_F1 = 0.8525

f1_difference = (
    best_validation_f1
    - BASELINE_MACRO_F1
)

relative_f1_improvement = (
    f1_difference
    / BASELINE_MACRO_F1
    * 100
)

In [26]:
results_path = (
    output_dir
    / "fine_tuned_results.txt"
)

results_text = f"""Transformers Day 5 — Fine-tuning Results
=========================================

Model: {MODEL_NAME}
Train samples: {len(train_dataset)}
Validation samples: {len(validation_dataset)}
Epochs: {NUM_EPOCHS}
Batch size: {BATCH_SIZE}
Learning rate: {LEARNING_RATE}

Best epoch: {best_epoch}
Validation Accuracy: {best_validation_accuracy_check:.4f}
Validation Macro F1: {best_validation_f1_check:.4f}

Baseline Accuracy: {BASELINE_ACCURACY:.4f}
Baseline Macro F1: {BASELINE_MACRO_F1:.4f}

Absolute F1 improvement: {f1_difference:.4f}
Relative F1 improvement: {relative_f1_improvement:.2f}%
"""

results_path.write_text(
    results_text,
    encoding="utf-8",
)

print(results_text)
print("Results:", results_path.resolve())
print("File exists:", results_path.exists())

Transformers Day 5 — Fine-tuning Results

Model: distilbert-base-uncased
Train samples: 1600
Validation samples: 400
Epochs: 3
Batch size: 16
Learning rate: 2e-05

Best epoch: 3
Validation Accuracy: 0.8900
Validation Macro F1: 0.8900

Baseline Accuracy: 0.8525
Baseline Macro F1: 0.8525

Absolute F1 improvement: 0.0375
Relative F1 improvement: 4.40%

Results: C:\Users\User\Projects\transformers_overall_fix\outputs\fine_tuned\fine_tuned_results.txt
File exists: True


In [27]:
model_files = sorted(
    fine_tuned_dir.glob(
        "model-*.safetensors"
    )
)

print("Model shards:")

for model_file in model_files:
    size_mb = (
        model_file.stat().st_size
        / 1024**2
    )

    print(
        model_file.name,
        f"{size_mb:.2f} MB",
    )

print(
    "Index file exists:",
    (
        fine_tuned_dir
        / "model.safetensors.index.json"
    ).exists(),
)

Model shards:
model-00001-of-00005.safetensors 89.42 MB
model-00002-of-00005.safetensors 46.58 MB
model-00003-of-00005.safetensors 45.08 MB
model-00004-of-00005.safetensors 47.31 MB
model-00005-of-00005.safetensors 27.04 MB
Index file exists: True
